# Run life of heavy-oil PCP wells

Every chart comes from `src/plots.py`, the same functions the app uses, so the numbers here always match the app and the README figures.

In [ ]:
import sys
from pathlib import Path

# The notebook runs from notebooks/, so add the repo root to the import path
# to make `src` importable.
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))

import pandas as pd

from src import config, plots

d = plots.load_all(config.OUTPUTS)
events = pd.read_parquet(config.EVENTS)
runs = pd.read_parquet(config.RUNS)
watchlist = pd.read_csv(config.WELLS_OUT / "well_watchlist.csv")

## 1. How big is the problem?

### Downhole failures every year

In [ ]:
fig = plots.failures_per_year(events)
fig.show()

### A small share of wells carries most of the damage

In [ ]:
fig = plots.concentration(d["pareto"])
fig.show()

### What failures cost each year

In [ ]:
fig = plots.cost_per_year(d["sizing"])
fig.show()

## 2. What drives run life?

Kaplan-Meier curves, not histograms: runs still going when the data ends are counted up to that point instead of being dropped.

### Failure history

In [ ]:
fig = plots.run_life(d["km_curves"], d["km_summary"], "history")
fig.show()

### Pump speed

In [ ]:
fig = plots.run_life(d["km_curves"], d["km_summary"], "rpm")
fig.show()

### Water cut

In [ ]:
fig = plots.run_life(d["km_curves"], d["km_summary"], "water_cut")
fig.show()

### All factors together (Cox model)

In [ ]:
fig = plots.risk_drivers(d["cox_results"])
fig.show()

## 3. Is a workover worth it?

### Run life by last repair

In [ ]:
fig = plots.run_life(d["km_curves"], d["km_summary"], "repair")
fig.show()

### Payback against how often a well fails

In [ ]:
inputs = plots.payback_inputs(runs, events, d["cox_results"])
fig = plots.workover_payback(inputs, oil_value=config.OIL_VALUE_PER_BBL)
fig.show()

## 4. Watchlist

### Where the open wells sit on 90-day risk

In [ ]:
fig = plots.risk_distribution(watchlist, top_n=config.TOP_K)
fig.show()

## 5. Does it work?

### Top 50 on the official failure log

In [ ]:
fig = plots.model_vs_rule(d["model_eval"], "official")
fig.show()

### Top 50 on held-out runs

In [ ]:
fig = plots.model_vs_rule(d["model_eval"], "test")
fig.show()

### Calibration

In [ ]:
fig = plots.calibration(d["calibration"])
fig.show()

### Rebuilt failures against the official log

In [ ]:
fig = plots.what_breaks(d["event_validation"])
fig.show()